# Tables 2 & 3 — Cluster Membership Logistic Regression with FDR Sensitivity Analysis

This notebook regenerates manuscript Tables 2 (Race/Ethnicity) and 3 (Age) showing **both unadjusted and FDR-adjusted p-values** in each cell, in response to Reviewer 1, Comment 8.

The output preserves the manuscript's exact column headers and per-table cluster names so the result is drop-in compatible with the existing manuscript tables.

**Inputs:**
* `MANUSCRIPT_DOCX` — the manuscript .docx file (source of unadjusted p-values, column headers, and cluster names).
* `ADJUSTED_CSV` — the regression output CSV with the FDR-adjusted p-values.

**Workflow:**
1. Open the .docx and extract Tables 2 and 3 directly from the document's table structure. Parse each cell to get OR/CI, unadjusted p, and significance stars. Capture each table's cluster names and column headers verbatim.
2. Cross-check: confirm every extracted OR/CI matches the corresponding row in the adjusted CSV.
3. Merge unadjusted and adjusted p-values on `(cluster_id, term_clean)`.
4. Render each output cell as three stacked lines: `OR(CI);` / `p=...;` / `pBH=...`.
5. Write a two-sheet Excel file using manuscript-matching headers and cluster names.

In [ ]:
# python-docx is required for reading .docx tables. Run once if not already installed:
# !pip install python-docx xlsxwriter

import pandas as pd
import numpy as np
import os
import re
import docx           # python-docx
import xlsxwriter     # enforce xlsxwriter use for Excel export

In [ ]:
# =====================================================================
# Config
# =====================================================================

MANUSCRIPT_DOCX = "/content/DOM_T2D_Manuscript_v2026-04-16.docx"
ADJUSTED_CSV    = "/content/logistic_regression_for_cluster_multivariable_20260512.csv"
OUT_PATH        = "/content/Regression_ORCI_Wide_Tables_with_FDR.xlsx"

SAVE_UNADJUSTED_CSV = True
UNADJUSTED_CSV_PATH = "/content/logistic_regression_unadjusted_from_manuscript.csv"

# python-docx is 0-indexed. Table 1 (patient characteristics) = index 0,
# Table 2 (Race) = index 1, Table 3 (Age) = index 2.
DOCX_RACE_TABLE_INDEX = 1
DOCX_AGE_TABLE_INDEX  = 2

# Display label for the adjusted p-value within each cell
ADJ_LABEL = "pBH"
ADJ_METHOD_NOTE = "Benjamini-Hochberg false discovery rate (BH-FDR)"

# Group ordering for sorting rows — matches the manuscript's row order
desired_group_order = ["Monotherapy", "Dual Therapy", "Complex Therapy", "Variant Therapy", "GLP-1 Therapy"]

# Per-table config: maps the manuscript table's data-column position
# to (term_clean used in CSV, manuscript-style column header).
# The order of these dicts is preserved in the output.
race_columns = {
    2: ("Race: Black",    "Black, NH"),
    3: ("Race: Hispanic", "Hispanic"),
    4: ("Race: Asian",    "Asian, NH"),
    5: ("Race: Other",    "Other"),
}
age_columns = {
    2: ("Age: 55-64 vs <55", "55-64 years"),
    3: ("Age: 65+ vs <55",   "> 65 years"),
}

# Manuscript-exact first-column header per table
RACE_FIRST_COL = "Cluster"  # Table 2 header
AGE_FIRST_COL  = "Group"    # Table 3 header

# Manuscript-exact table titles (for the Excel banner)
RACE_TABLE_TITLE = "Table 2. Odds of Membership in Each Cluster by Race and Ethnicity"
AGE_TABLE_TITLE  = "Table 3. Odds of Membership in Each Trajectory Cluster by Age Category"

rows_to_keep = 30

In [ ]:
# =====================================================================
# Step 1: Extract unadjusted p-values from manuscript Tables 2 and 3.
# Each row also captures the cluster name verbatim from the manuscript,
# since Table 2 and Table 3 use slightly different names for three GLP-1 clusters.
# =====================================================================

CLUSTER_PREFIX_TO_GROUP = {
    "Mono:":    "Monotherapy",
    "Dual:":    "Dual Therapy",
    "Complex:": "Complex Therapy",
    "Variant:": "Variant Therapy",
    "GLP-1:":   "GLP-1 Therapy",
}

_CELL_RE = re.compile(
    r'(?P<or>\d+\.\d+)\s*\(\s*(?P<lo>[\d.]+)\s*-\s*(?P<hi>[\d.]+)\s*\)\s*;\s*'
    r'p\s*=\s*(?P<pval><?\s*\d*\.?\d+)\s*(?P<sig>\*+)?\s*$'
)

def _infer_group(cluster_name):
    for prefix, group in CLUSTER_PREFIX_TO_GROUP.items():
        if cluster_name.startswith(prefix):
            return group
    raise ValueError(f"Unknown cluster prefix: {cluster_name!r}")

def _parse_manuscript_cell(cell_text):
    s = cell_text.strip()
    m = _CELL_RE.match(s)
    if not m:
        raise ValueError(f"Could not parse manuscript cell: {s!r}")
    or_ci = f"{m['or']} ({m['lo']}-{m['hi']})"
    pval  = m['pval'].replace(" ", "")
    sig   = m['sig'] or ""
    return or_ci, pval, sig

def extract_unadjusted_from_docx(docx_path, cluster_lookup,
                                 race_idx=DOCX_RACE_TABLE_INDEX,
                                 age_idx=DOCX_AGE_TABLE_INDEX):
    """Open the manuscript .docx and return a long-format DataFrame of
    unadjusted p-values. Each row carries `cluster_name_manuscript` taken
    verbatim from the source table's first column."""
    doc = docx.Document(docx_path)
    if len(doc.tables) <= max(race_idx, age_idx):
        raise ValueError(
            f"Manuscript has only {len(doc.tables)} tables; expected at least "
            f"{max(race_idx, age_idx) + 1}."
        )

    table_specs = [
        (doc.tables[race_idx], race_columns, "Race"),
        (doc.tables[age_idx],  age_columns,  "Age"),
    ]

    records = []
    for table_obj, col_map, label in table_specs:
        n_data_rows = len(table_obj.rows) - 1
        if n_data_rows != rows_to_keep:
            print(f"[Warning] {label} table has {n_data_rows} data rows; expected {rows_to_keep}.")

        for row_idx in range(1, len(table_obj.rows)):
            row = table_obj.rows[row_idx]
            cluster_name = row.cells[0].text.strip()
            group_rank   = int(row.cells[1].text.strip())
            group        = _infer_group(cluster_name)

            key = (group, group_rank)
            if key not in cluster_lookup.index:
                raise KeyError(
                    f"Manuscript ({group}, rank={group_rank}) not found in adjusted CSV — "
                    f"check that ADJUSTED_CSV is from the same regression run."
                )
            cluster_id = int(cluster_lookup.loc[key, "cluster_id"])
            n_patients = int(cluster_lookup.loc[key, "n_patients"])

            for col_idx, (term_clean, _column_header) in col_map.items():
                cell_text = row.cells[col_idx].text
                or_ci, pval, sig = _parse_manuscript_cell(cell_text)
                records.append({
                    "cluster_id":               cluster_id,
                    "n_patients":               n_patients,
                    "Group":                    group,
                    "group_rank":               group_rank,
                    "cluster_name_manuscript":  cluster_name,
                    "predictor_label":          "Multivariable model",
                    "term_clean":               term_clean,
                    "OR_CI":                    or_ci,
                    "p.value.fmt":              pval,
                    "sig":                      sig,
                })

    return pd.DataFrame(records)

adj_df = pd.read_csv(ADJUSTED_CSV)
cluster_lookup = (adj_df[["Group", "group_rank", "cluster_id", "n_patients"]]
                  .drop_duplicates()
                  .set_index(["Group", "group_rank"]))

unadj_df = extract_unadjusted_from_docx(MANUSCRIPT_DOCX, cluster_lookup)
print(f"Extracted {len(unadj_df)} rows from manuscript Tables 2 and 3 "
      f"({unadj_df['cluster_id'].nunique()} unique clusters, "
      f"{unadj_df['term_clean'].nunique()} unique terms).")

if SAVE_UNADJUSTED_CSV:
    os.makedirs(os.path.dirname(UNADJUSTED_CSV_PATH) or ".", exist_ok=True)
    unadj_df.to_csv(UNADJUSTED_CSV_PATH, index=False)
    print(f"Saved tidy unadjusted-p CSV: {UNADJUSTED_CSV_PATH}")

unadj_df.head(6)

In [ ]:
# =====================================================================
# Step 2: Sanity check — every OR/CI from the manuscript must match the
# corresponding row in the adjusted CSV.
# =====================================================================

def _normalize_orci(s):
    return str(s).replace(" ", "").replace("\u2013", "-")

merged_check = unadj_df.merge(
    adj_df[["cluster_id", "term_clean", "OR_CI"]].rename(columns={"OR_CI": "OR_CI_adj"}),
    on=["cluster_id", "term_clean"], how="left",
)
merged_check["match"] = merged_check.apply(
    lambda r: _normalize_orci(r["OR_CI"]) == _normalize_orci(r["OR_CI_adj"]), axis=1
)
n_total = len(merged_check)
n_match = int(merged_check["match"].sum())
print(f"OR_CI consistency: {n_match}/{n_total} cells match between manuscript and adjusted CSV.")

if n_match != n_total:
    print("\nMISMATCHES detected:")
    print(merged_check.loc[~merged_check["match"],
                          ["cluster_id", "term_clean", "OR_CI", "OR_CI_adj"]].to_string(index=False))
    raise AssertionError("OR_CI mismatch — investigate before proceeding.")

In [ ]:
# =====================================================================
# Step 3: Merge unadjusted and adjusted p-values on (cluster_id, term_clean).
# =====================================================================

adj_slim = adj_df[["cluster_id", "term_clean", "p.value.adjusted.fmt", "sig"]].rename(
    columns={"sig": "sig.adjusted"}
)

regression_df = unadj_df.merge(adj_slim, on=["cluster_id", "term_clean"], how="left")

n_missing = regression_df["p.value.adjusted.fmt"].isna().sum()
if n_missing:
    raise AssertionError(f"{n_missing} rows have no adjusted p-value after merge — check inputs.")

print(f"Merged dataframe: {len(regression_df)} rows, columns: {list(regression_df.columns)}")
regression_df.head(6)

In [ ]:
# =====================================================================
# Step 4: Cell renderer — 3-line stacked format:
#   OR(CI);
#   p=<unadjusted>;
#   pBH=<adjusted>
# =====================================================================

def _normalize_orci_for_display(s):
    if pd.isna(s):
        return ""
    s = str(s).strip()
    s = s.replace(" (", "(")
    s = s.replace("\u2013", "-")
    s = s.replace("( ", "(").replace(" )", ")")
    return s

def _p_to_float(p):
    if pd.isna(p):
        return float("nan")
    s = str(p).strip()
    if not s:
        return float("nan")
    if s.startswith("<"):
        try:
            return float(s.lstrip("<")) - 1e-9
        except ValueError:
            return 0.0005
    try:
        return float(s)
    except ValueError:
        return float("nan")

def _sig_stars(p):
    pf = _p_to_float(p)
    if np.isnan(pf): return ""
    if pf < 0.001: return "***"
    if pf < 0.01:  return "**"
    if pf < 0.05:  return "*"
    return ""

def _sig_for(row, p_col, sig_col):
    if sig_col in row and pd.notna(row[sig_col]) and str(row[sig_col]).strip() != "":
        return str(row[sig_col]).strip()
    return _sig_stars(row[p_col])

def make_display_cell_stacked(row):
    orci = _normalize_orci_for_display(row["OR_CI"]) if pd.notna(row["OR_CI"]) else ""
    if not orci:
        return "\u2014"

    p_unadj = ("" if pd.isna(row["p.value.fmt"]) else str(row["p.value.fmt"]).strip())
    p_adj   = ("" if pd.isna(row["p.value.adjusted.fmt"]) else str(row["p.value.adjusted.fmt"]).strip())

    sig_unadj = _sig_for(row, "p.value.fmt",          "sig")
    sig_adj   = _sig_for(row, "p.value.adjusted.fmt", "sig.adjusted")

    lines = [f"{orci};"]
    if p_unadj: lines.append(f"p={p_unadj}{sig_unadj};")
    if p_adj:   lines.append(f"{ADJ_LABEL}={p_adj}{sig_adj}")
    if lines and lines[-1].endswith(";"):
        lines[-1] = lines[-1].rstrip(";")
    return "\n".join(lines)

In [ ]:
# =====================================================================
# Step 5: Build wide tables using manuscript-exact column headers and
# per-table cluster names.
#
# Output column order: <first_col_name> | # | <data columns>
# =====================================================================

def build_wide_sheet(df, col_map, first_col_name, title_label):
    wanted_terms   = [term for term, _hdr in col_map.values()]
    col_header_for = {term: hdr for (term, hdr) in col_map.values()}
    ordered_headers = [col_header_for[t] for t in wanted_terms]

    d = df[df["Group"].isin(desired_group_order) & df["term_clean"].isin(wanted_terms)].copy()
    if d.empty:
        raise ValueError(f"[{title_label}] No rows found for requested terms.")

    d["Group"]     = pd.Categorical(d["Group"], categories=desired_group_order, ordered=True)
    d["Hash"]      = pd.to_numeric(d["group_rank"], errors="coerce").astype("Int64")
    d["ColHeader"] = d["term_clean"].map(col_header_for)
    d["Display"]   = d.apply(make_display_cell_stacked, axis=1)

    d = d.sort_values(["Group", "Hash", "ColHeader"]).reset_index(drop=True)

    # Pivot using the manuscript's cluster name (per-table) plus Group / # for sorting
    wide = d.pivot_table(
        index=["Group", "Hash", "cluster_name_manuscript"],
        columns="ColHeader", values="Display", aggfunc="first",
    )
    wide = wide.reindex(columns=ordered_headers)
    wide = wide.reset_index().sort_values(["Group", "Hash"]).reset_index(drop=True)
    wide[ordered_headers] = wide[ordered_headers].fillna("\u2014")

    if len(wide) < rows_to_keep:
        print(f"[{title_label}] Note: only {len(wide)} rows available after filtering.")
    wide = wide.head(rows_to_keep).copy()

    # Drop the sort-only Group column BEFORE renaming, so when first_col_name itself
    # is 'Group' (Age table) we don't end up with two columns named 'Group'.
    wide = wide.drop(columns=["Group"])
    wide = wide.rename(columns={"cluster_name_manuscript": first_col_name, "Hash": "#"})
    wide = wide[[first_col_name, "#"] + ordered_headers]
    return wide

race_wide = build_wide_sheet(regression_df, race_columns, RACE_FIRST_COL, "Race")
age_wide  = build_wide_sheet(regression_df, age_columns,  AGE_FIRST_COL,  "Age")

print(f"Race table shape: {race_wide.shape}")
print(f"Race columns: {list(race_wide.columns)}")
print(f"Age  table shape: {age_wide.shape}")
print(f"Age  columns: {list(age_wide.columns)}")
race_wide.head(5)

In [ ]:
# =====================================================================
# Step 6: Export Excel — manuscript-matching headers; auto table note.
# =====================================================================

def save_two_wide_sheets(race_df, age_df, xlsx_path=OUT_PATH,
                        body_font_size=10, header_font_size=11, title_font_size=12):
    os.makedirs(os.path.dirname(xlsx_path) or ".", exist_ok=True)

    table_note = (
        f"Each cell contains OR (95% CI); p = unadjusted p-value from multivariable logistic regression; "
        f"{ADJ_LABEL} = {ADJ_METHOD_NOTE} adjusted p-value. "
        f"*** p<0.001, ** p<0.01, * p<0.05."
    )

    with pd.ExcelWriter(xlsx_path, engine="xlsxwriter") as writer:
        startrow = 2

        for sheet_name, df_, banner in [
            ("Race ORs", race_df, RACE_TABLE_TITLE),
            ("Age ORs",  age_df,  AGE_TABLE_TITLE),
        ]:
            df_.to_excel(writer, sheet_name=sheet_name, index=False, startrow=startrow)
            wb = writer.book
            ws = writer.sheets[sheet_name]

            title_fmt  = wb.add_format({"bold": True, "font_size": title_font_size, "align": "left"})
            header_fmt = wb.add_format({"bold": True, "font_size": header_font_size, "bottom": 1,
                                        "text_wrap": True, "valign": "top"})
            body_fmt   = wb.add_format({"font_size": body_font_size,
                                        "text_wrap": True, "valign": "top"})
            note_fmt   = wb.add_format({"font_size": body_font_size - 1, "italic": True,
                                        "text_wrap": True, "valign": "top"})

            n_cols = df_.shape[1]
            n_rows = df_.shape[0]
            ws.merge_range(0, 0, 0, n_cols - 1, banner, title_fmt)

            for c, name in enumerate(df_.columns):
                ws.write(startrow, c, name, header_fmt)

            for c, name in enumerate(df_.columns):
                if name in ("Cluster", "Group"):
                    ws.set_column(c, c, 32, body_fmt)
                elif name == "#":
                    ws.set_column(c, c, 5, body_fmt)
                else:
                    ws.set_column(c, c, 24, body_fmt)

            for r in range(startrow + 1, startrow + 1 + n_rows):
                ws.set_row(r, 46, body_fmt)

            note_row = startrow + n_rows + 1
            ws.merge_range(note_row, 0, note_row, n_cols - 1, table_note, note_fmt)
            ws.set_row(note_row, 30)

            ws.set_landscape()
            ws.set_paper(1)  # Letter
            ws.set_margins(left=0.5, right=0.5, top=0.6, bottom=0.6)
            ws.print_area(0, 0, note_row, n_cols - 1)
            ws.fit_to_pages(1, 1)

    print(f"\u2705 Exported Excel file with both tables:\n{xlsx_path}")

save_two_wide_sheets(race_wide, age_wide, xlsx_path=OUT_PATH)

## Notes for the manuscript

The Excel column structure now matches the manuscript exactly, so the data cells can be copy-pasted over the existing manuscript Tables 2 and 3.

Three remaining manuscript edits:

1. **Table notes** (currently around lines 414 and 451 of the manuscript). Append definition of `pBH`. Suggested addition to each Note:
   > *In each cell, `p` is the unadjusted p-value and `pBH` is the Benjamini-Hochberg false discovery rate-adjusted p-value. Significance stars apply independently to each: \*\*\* p<0.001, \*\* p<0.01, \* p<0.05.*
2. **Methods statement** (around line 187). Replace with:
   > *Because analyses were exploratory, unadjusted p-values are reported as primary; Benjamini-Hochberg false discovery rate-adjusted p-values are reported alongside as a sensitivity analysis.*
3. **Results, line 227** — the only in-text quoted p-value (`p=0.039`) loses significance after BH adjustment (`pBH=0.104`), so the claim about Hispanic patients in late-transition GLP-1 trajectories may need softening.